# Decision thresholds

Precision, recall, and F1 across decision thresholds, using the saved held-out probabilities.

In [ ]:
from shared import paths
from IPython.display import display


## Setup

In [ ]:
paths.prepare_directories()


## Binary reference detection

In [ ]:
import pandas as pd
import ast
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support

path = str(paths.sentence_run(3) / 'general/all_folds_predictions.csv')
df = pd.read_csv(path)
df['true_labels'] = df['true_labels'].apply(ast.literal_eval)
df['probabilities'] = df['probabilities'].apply(ast.literal_eval)
thresholds = [round(x, 2) for x in [i / 100 for i in range(5, 100, 5)]]
ref_metrics = {'threshold': [], 'precision': [], 'recall': [], 'f1': []}
noref_metrics = {'threshold': [], 'precision': [], 'recall': [], 'f1': []}
for threshold in thresholds:
    pred_any = df['probabilities'].apply(lambda probs: int(any((p >= threshold for p in probs))))
    true_any = df['true_labels'].apply(lambda labels: int(any((l == 1 for l in labels))))
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_any,
        pred_any,
        average=None,
        labels=[1, 0],
        zero_division=0,
    )
    ref_metrics['threshold'].append(threshold)
    ref_metrics['precision'].append(precision[0])
    ref_metrics['recall'].append(recall[0])
    ref_metrics['f1'].append(f1[0])
    noref_metrics['threshold'].append(threshold)
    noref_metrics['precision'].append(precision[1])
    noref_metrics['recall'].append(recall[1])
    noref_metrics['f1'].append(f1[1])
sns.set_theme(style='darkgrid', palette='colorblind')
plt.rcParams.update(
    {
        'axes.titlesize': 16,
        'axes.labelsize': 14,
        'legend.fontsize': 12,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'lines.linewidth': 2,
        'lines.markersize': 7,
    },
)


def plot_metrics(metrics_dict, title, filename=None):
    df_metrics = pd.DataFrame(metrics_dict)
    best_idx = df_metrics['f1'].idxmax()
    best_threshold = df_metrics.loc[best_idx, 'threshold']
    best_f1 = df_metrics.loc[best_idx, 'f1']
    plt.figure(figsize=(10, 6))
    sns.lineplot(x='threshold', y='precision', data=df_metrics, label='Precision', marker='o')
    sns.lineplot(x='threshold', y='recall', data=df_metrics, label='Recall', marker='s')
    sns.lineplot(x='threshold', y='f1', data=df_metrics, label='F1 Score', marker='^')
    plt.axvline(
        x=best_threshold,
        linestyle='--',
        alpha=0.6,
        label=f'Best F1 = {best_f1:.3f} @ thr = {best_threshold:.2f}',
    )
    plt.xlabel('Threshold')
    plt.ylabel('Score')
    plt.ylim(0.7, 1.01)
    plt.xticks(df_metrics['threshold'])
    plt.legend(loc='lower left', frameon=True)
    plt.tight_layout()
    if filename:
        plt.savefig(filename, dpi=300)
    plt.show()
    print(f'\n=== {title} ===')
    print(df_metrics.to_string(index=False, float_format='%.4f'))
    print(f'\nBest threshold: {best_threshold:.2f} (F1 = {best_f1:.4f})')
plot_metrics(ref_metrics, 'Performance for Ref Class Across Thresholds')
plot_metrics(noref_metrics, 'Performance for NoRef Class Across Thresholds')


## Low-threshold detail

In [ ]:
import pandas as pd
import ast
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support

path = str(paths.sentence_run(3) / 'general/all_folds_predictions.csv')
df = pd.read_csv(path)
df['true_labels'] = df['true_labels'].apply(ast.literal_eval)
df['probabilities'] = df['probabilities'].apply(ast.literal_eval)
sns.set_theme(style='darkgrid', palette='colorblind')
plt.rcParams.update(
    {
        'axes.titlesize': 16,
        'axes.labelsize': 14,
        'legend.fontsize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 11,
        'lines.linewidth': 2,
        'lines.markersize': 7,
    },
)
thresholds = [round(x / 1000, 4) for x in range(1, 51)]
ref_metrics = {'threshold': [], 'precision': [], 'recall': [], 'f1': []}
for threshold in thresholds:
    pred_any = df['probabilities'].apply(lambda probs: int(any((p >= threshold for p in probs))))
    true_any = df['true_labels'].apply(lambda labels: int(any((l == 1 for l in labels))))
    precision, recall, f1, _ = precision_recall_fscore_support(true_any, pred_any, average=None, labels=[1], zero_division=0)
    ref_metrics['threshold'].append(threshold)
    ref_metrics['precision'].append(precision[0])
    ref_metrics['recall'].append(recall[0])
    ref_metrics['f1'].append(f1[0])
plt.figure(figsize=(12, 6))
sns.lineplot(x=ref_metrics['threshold'], y=ref_metrics['precision'], label='Precision', marker='o')
sns.lineplot(x=ref_metrics['threshold'], y=ref_metrics['recall'], label='Recall', marker='s')
sns.lineplot(x=ref_metrics['threshold'], y=ref_metrics['f1'], label='F1 Score', marker='^')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.ylim(0.65, 1.01)
xticks_to_show = [0.001] + [round(x / 1000, 3) for x in range(5, 51, 5)]
plt.xticks(xticks_to_show)
plt.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.show()


## Multilabel categorisation

In [ ]:
import ast
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import os

INPUT_CSV = str(paths.sentence_run(4) / 'general/all_folds_predictions.csv')
THRESHOLDS = [round(x, 2) for x in [i / 100 for i in range(5, 100, 5)]]
CATEGORIES = [
    'Editorial and Publishing Processes',
    'Elements of Fictional Craft',
    'Financial and Legal Aspects',
    'Gratitude and Acknowledgment',
    'Impact and Reception',
    'Inspirations and Influences',
    'Marketing and Promotion',
    'Quality and Evaluation',
    'Writing Process and Creative Insight',
]
NUM_CATEGORIES = len(CATEGORIES)
df = pd.read_csv(INPUT_CSV)
df['true_labels'] = df['true_labels'].apply(ast.literal_eval)
df['probabilities'] = df['probabilities'].apply(ast.literal_eval)
y_true = np.array(df['true_labels'].tolist(), dtype=int)
y_prob = np.array(df['probabilities'].tolist(), dtype=float)


def binarise(probs, thr):
    return (probs >= thr).astype(int)


def per_category_metrics(y_true_bin, y_pred_bin):
    p, r, f1, support = precision_recall_fscore_support(y_true_bin, y_pred_bin, average=None, zero_division=0)
    return (p, r, f1, support)


def macro_from_vectors(p_vec, r_vec, f1_vec):
    return (float(np.mean(p_vec)), float(np.mean(r_vec)), float(np.mean(f1_vec)))


def weighted_from_vectors(p_vec, r_vec, f1_vec, weights):
    W = float(np.sum(weights))
    if W == 0:
        return (0.0, 0.0, 0.0)
    P_w = float(np.sum(weights * p_vec) / W)
    R_w = float(np.sum(weights * r_vec) / W)
    F1_w = float(np.sum(weights * f1_vec) / W)
    return (P_w, R_w, F1_w)
instances = y_true.sum(axis=0).astype(float)
total_instances = float(instances.sum())
rows = []
per_cat_scores = {c: {'threshold': [], 'precision': [], 'recall': [], 'f1': []} for c in range(NUM_CATEGORIES)}
for thr in THRESHOLDS:
    y_pred = binarise(y_prob, thr)
    p_c, r_c, f_c, _ = per_category_metrics(y_true, y_pred)
    p_macro, r_macro, f_macro = macro_from_vectors(p_c, r_c, f_c)
    p_weighted, r_weighted, f_weighted = weighted_from_vectors(p_c, r_c, f_c, instances)
    rows.append(
        {
            'threshold': thr,
            'precision_macro': p_macro,
            'recall_macro': r_macro,
            'f1_macro': f_macro,
            'precision_weighted': p_weighted,
            'recall_weighted': r_weighted,
            'f1_weighted': f_weighted,
        },
    )
    for c in range(NUM_CATEGORIES):
        per_cat_scores[c]['threshold'].append(thr)
        per_cat_scores[c]['precision'].append(float(p_c[c]))
        per_cat_scores[c]['recall'].append(float(r_c[c]))
        per_cat_scores[c]['f1'].append(float(f_c[c]))
curves_df = pd.DataFrame(rows)
best_thr_macro = float(curves_df.loc[curves_df['f1_macro'].idxmax(), 'threshold'])
best_thr_weighted = float(curves_df.loc[curves_df['f1_weighted'].idxmax(), 'threshold'])
best_row_macro = curves_df.loc[curves_df['f1_macro'].idxmax()].to_dict()
best_row_weighted = curves_df.loc[curves_df['f1_weighted'].idxmax()].to_dict()
per_category_summary = []
for c, name in enumerate(CATEGORIES):
    cat_df = pd.DataFrame(per_cat_scores[c])
    best_idx = cat_df['f1'].idxmax()
    best_row = cat_df.loc[best_idx]
    per_category_summary.append(
        {
            'category': name,
            'best_threshold': float(best_row['threshold']),
            'precision_at_best': float(best_row['precision']),
            'recall_at_best': float(best_row['recall']),
            'f1_at_best': float(best_row['f1']),
        },
    )
per_category_df = pd.DataFrame(per_category_summary)
sns.set_theme(style='darkgrid', palette='colorblind')
plt.rcParams.update(
    {
        'axes.titlesize': 16,
        'axes.labelsize': 14,
        'legend.fontsize': 12,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'lines.linewidth': 2,
        'lines.markersize': 7,
    },
)
plt.figure(figsize=(10, 6))
sns.lineplot(
    x='threshold',
    y='precision_macro',
    data=curves_df,
    label='Precision (macro)',
    marker='o',
)
sns.lineplot(x='threshold', y='recall_macro', data=curves_df, label='Recall (macro)', marker='s')
sns.lineplot(x='threshold', y='f1_macro', data=curves_df, label='F1 (macro)', marker='^')
plt.axvline(
    x=best_thr_macro,
    linestyle='--',
    alpha=0.7,
    label=f'Best macro-F1: {best_thr_macro:.2f}',
)
plt.axvline(
    x=best_thr_weighted,
    linestyle='--',
    alpha=0.7,
    label=f'Best weighted-F1: {best_thr_weighted:.2f}',
)
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.ylim(0.0, 1.01)
plt.yticks(np.arange(0.0, 1.1, 0.1))
plt.xticks(curves_df['threshold'].values)
plt.legend(loc='lower left', frameon=True)
plt.tight_layout()
plt.show()
print('\n=== Macro & Weighted metrics per threshold ===')
print(curves_df.to_string(index=False, float_format='%.4f'))
print('\n=== Best threshold by averaging scheme ===')
print(
    f"Macro-F1 best @ tau={best_thr_macro:.2f} -> P={best_row_macro['precision_macro']:.4f} R={best_row_macro['recall_macro']:.4f} F1={best_row_macro['f1_macro']:.4f}",
)
print(
    f"Weighted-F1 best @ tau={best_thr_weighted:.2f} -> P={best_row_weighted['precision_weighted']:.4f} R={best_row_weighted['recall_weighted']:.4f} F1={best_row_weighted['f1_weighted']:.4f}",
)
print('\n=== Per-category optimal thresholds (argmax F1 per category) ===')
print(
    per_category_df.sort_values('f1_at_best', ascending=False).to_string(
        index=False,
        float_format='%.4f',
    ),
)
